# 06 — Run the confirmatory layer (CPU, minutes)
## THE BLIND BREAKS HERE

## 1. Verify the GPU(s)

In [ ]:
!nvidia-smi

## 2. Clone the repo

Set `PROBE_REPO_URL` to your public Git URL or `PROBE_REPO_DIR` to a local checkout.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = os.environ.get("PROBE_REPO_DIR", "/kaggle/working/probe-capacity-invariance")
if not (Path(REPO_DIR) / "pyproject.toml").is_file():
    repo_url = os.environ.get("PROBE_REPO_URL")
    if not repo_url:
        raise RuntimeError("Set PROBE_REPO_URL or PROBE_REPO_DIR before running this notebook.")
    subprocess.run(["git", "clone", repo_url, REPO_DIR], check=True)
os.chdir(REPO_DIR)


## 3. Install dependencies

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| device count:", torch.cuda.device_count())

In [ ]:


import shutil
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance"); INPUT = Path("/kaggle/input")
restored = 0
for src in list(INPUT.glob("*/results")) + list(INPUT.glob("*/probe-capacity-invariance/results")):
    for f in src.rglob("*"):
        if f.is_file() and f.suffix in (".npz", ".json", ".jsonl"):
            dst = REPO / "results" / f.relative_to(src)
            if not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, dst); restored += 1
print(f"restored {restored} prior result files")

## 5. Pre-flight — the realized grid must match A7 exactly

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/probe-capacity-invariance")
from src.probes.hypotheses import discover_cells, EXPECTED_CELLS, PRE_VERDICT_EXCLUSIONS
cells = sorted(p.name for p in discover_cells("/kaggle/working/probe-capacity-invariance/results/probes_pinned"))
print("realized grid:", cells)
print("expected     :", sorted(EXPECTED_CELLS))
print("excluded     :", list(PRE_VERDICT_EXCLUSIONS))
assert set(cells) == set(EXPECTED_CELLS), "grid mismatch -- resolve BEFORE unblinding"
print("\nOK. The next cell breaks the blind.")

## 6. Confirm A8 repairs are present in every cell (still blind)

In [ ]:
import numpy as np, json
from pathlib import Path
for cell in sorted(EXPECTED_CELLS):
    d = Path("/kaggle/working/probe-capacity-invariance") / "results/probes_pinned" / cell
    z = np.load(d / "stacks.npz"); m = json.loads((d / "meta.json").read_text())
    assert "random_projector" in z.files, f"{cell}: re-run notebook 03 first"
    assert m.get("encoder_ckpts"), f"{cell}: provenance missing, re-run notebook 03"
    print(f"{cell:16s} OK  probe_train={m['probe_train_size']}")

## 7. Point of no return

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.hypotheses \
    --root results/probes_pinned --out results/hypotheses/hypothesis_report.json

## 8. Figures

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.eval.figures \
    --root results/probes_pinned --out results/figures/real

In [ ]:



import shutil
from pathlib import Path
src = Path("/kaggle/working/probe-capacity-invariance/results"); dst = Path("/kaggle/working/results")
shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst)
print(f"persisted {sum(1 for _ in dst.rglob('*') if _.is_file())} files "
      f"-> click 'Save Version' now")